In [ ]:

from pypdf import PdfReader
from sentence_transformers import SentenceTransformer
from transformers import pipeline
import faiss, numpy as np

# Load document
text=''.join(page.extract_text() or '' for page in PdfReader('week7.pdf').pages)

# Split text
chunks=[text[i:i+500] for i in range(0,len(text),500)]

# Embeddings
embedder=SentenceTransformer('all-MiniLM-L6-v2')
embeddings=embedder.encode(chunks).astype('float32')

# Vector database
index=faiss.IndexFlatL2(embeddings.shape[1])
index.add(embeddings)

# LLM
llm=pipeline('text2text-generation',model='google/flan-t5-base')

# User query
question=input('Enter your question: ')
q=embedder.encode([question]).astype('float32')
_,I=index.search(q,3)
context=' '.join(chunks[i] for i in I[0])

prompt=f'Context:\n{context}\n\nQuestion: {question}\nAnswer:'
print(llm(prompt,max_new_tokens=80)[0]['generated_text'])